# YOLOv26 Anomaly Detection — UCF Crime Dataset (Local Training)

Train a custom **YOLOv26** model for real-time CCTV anomaly detection using the
[UCF Crime Dataset](https://www.kaggle.com/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance).

**Dataset layout (after manual download):**
```
data/
├── abuse/             ← 50 video files (.mp4 / .avi)
├── arrest/
├── arson/
├── assault/
├── burglary/
├── explosion/
├── fighting/
├── normal/
├── roadaccidents/
├── robbery/
├── shooting/
├── shoplifting/
├── stealing/
├── vandalism/
├── train.csv          ← train split (video paths + labels)
└── test.csv           ← test  split (video paths + labels)
```

> **Task type:** Video-Classification → frames extracted → trained as **YOLO Classify**  
> Because this dataset has *no* bounding-box annotations, we use `task='classify'`.  
> Frames from each video are extracted and placed under the corresponding class folder.

## 1. Environment Setup & GPU Check

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU — training will be slow. Consider using a GPU.")

# Install / upgrade required packages
# (run this cell once; comment out after first run to save time)
import subprocess, sys

packages = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu",
    "-U ultralytics",
    "supervision",
    "pandas",
    "opencv-python",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]

# Uncomment the block below to install / upgrade dependencies:
# for pkg in packages:
#     subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkg.split())

## 2. Import Required Libraries

In [ ]:
import os
import shutil
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tqdm import tqdm

from sklearn.model_selection import train_test_split

from ultralytics import YOLO

print("All libraries imported successfully.")
print(f"Ultralytics version : {__import__('ultralytics').__version__}")

## 3. Paths & Configuration

Set `DATASET_ROOT` to the folder where you extracted the Kaggle download.  
Everything else is derived automatically.

In [ ]:
# ─── USER CONFIG ─────────────────────────────────────────────────────────────
# Change this to wherever you extracted the Kaggle dataset
DATASET_ROOT = Path("/path/to/your/dataset/data")   # ← EDIT THIS

# Notebook root (one level up from this file)
NOTEBOOK_DIR  = Path("__file__").resolve().parent if "__file__" in dir() else Path.cwd()
PROJECT_ROOT  = NOTEBOOK_DIR.parent.parent           # custom_models/../ == project root

# Where to store extracted frames (YOLO classification layout)
FRAMES_DIR    = PROJECT_ROOT / "custom_models" / "ucf_crime_frames"

# YOLO training output
RUNS_DIR      = PROJECT_ROOT / "custom_models" / "ucf_crime_runs"

# Final model output
MODEL_OUT_DIR = PROJECT_ROOT / "custom_models" / "ucf_crime_weights"

# Base YOLO26 weights (nano — fastest). Change to yolo26s.pt / yolo26m.pt for higher accuracy.
YOLO26_WEIGHTS = str(PROJECT_ROOT / "yolo26n.pt")

# Training hyper-parameters
EPOCHS         = 50
IMG_SIZE       = 224       # classification tasks typically use 224
BATCH_SIZE     = 16
FRAMES_PER_VID = 10        # frames to extract per video
DEVICE         = 0 if torch.cuda.is_available() else "cpu"
VAL_SPLIT      = 0.20      # fraction of TRAIN set used as validation
RANDOM_SEED    = 42

# ─────────────────────────────────────────────────────────────────────────────
for d in [FRAMES_DIR, RUNS_DIR, MODEL_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATASET_ROOT  :", DATASET_ROOT)
print("FRAMES_DIR    :", FRAMES_DIR)
print("RUNS_DIR      :", RUNS_DIR)
print("MODEL_OUT_DIR :", MODEL_OUT_DIR)
print("YOLO weights  :", YOLO26_WEIGHTS)
print("Device        :", DEVICE)

## 4. Dataset Exploration & CSV Analysis

In [ ]:
train_csv_path = DATASET_ROOT / "train.csv"
test_csv_path  = DATASET_ROOT / "test.csv"

# ── Load CSVs ──────────────────────────────────────────────────────────────
train_df = pd.read_csv(train_csv_path, header=None)
test_df  = pd.read_csv(test_csv_path,  header=None)

print("=" * 55)
print("TRAIN CSV")
print(f"  Shape   : {train_df.shape}")
print(f"  Columns : {list(train_df.columns)}")
print(f"  Nulls   :\n{train_df.isnull().sum()}")
print()
print("TEST CSV")
print(f"  Shape   : {test_df.shape}")
print(f"  Columns : {list(test_df.columns)}")
print(f"  Nulls   :\n{test_df.isnull().sum()}")
print("=" * 55)

# Preview
print("\nTrain sample rows:")
print(train_df.head(10).to_string())
print("\nTest sample rows:")
print(test_df.head(10).to_string())

In [ ]:
# ── Normalise column names ────────────────────────────────────────────────────
# The CSV typically has two columns: video_path  and  label
# Adapt the column indices below if your CSV differs.
train_df.columns = ["video_path", "label"][: len(train_df.columns)]
test_df.columns  = ["video_path", "label"][: len(test_df.columns)]

# ── Class distribution ────────────────────────────────────────────────────────
print("\nClass distribution in TRAIN:")
train_counts = train_df["label"].value_counts().sort_index()
print(train_counts.to_string())

print("\nClass distribution in TEST:")
test_counts = test_df["label"].value_counts().sort_index()
print(test_counts.to_string())

# ── Derive class list from directories in the dataset root ────────────────────
# (more reliable than reading from CSV if labels are folder names)
CLASSES = sorted([
    d.name for d in DATASET_ROOT.iterdir()
    if d.is_dir()
])
print(f"\nClasses detected ({len(CLASSES)}): {CLASSES}")

# ── Plot class distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, df, title in zip(axes, [train_df, test_df], ["Train", "Test"]):
    counts = df["label"].value_counts().sort_index()
    ax.barh(counts.index.astype(str), counts.values, color="steelblue")
    ax.set_title(f"{title} — {len(df)} videos")
    ax.set_xlabel("Video count")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Frame Extraction

Extract `FRAMES_PER_VID` evenly-spaced frames from every video listed in the
train / test CSVs and write them into a YOLO-classify directory layout:

```
ucf_crime_frames/
├── train/
│   ├── abuse/
│   │   ├── Abuse001_frame_000.jpg
│   │   └── ...
│   ├── arrest/
│   └── ...
└── val/
    ├── abuse/
    └── ...
```

In [ ]:
def extract_frames(video_path: Path, out_dir: Path, n_frames: int = 10) -> int:
    """
    Extract `n_frames` evenly-spaced frames from `video_path`
    and save them as JPEGs in `out_dir`.
    Returns the number of frames actually saved.
    """
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return 0

    indices = np.linspace(0, total - 1, min(n_frames, total), dtype=int)
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = 0
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            out_path = out_dir / f"{video_path.stem}_frame_{idx:04d}.jpg"
            cv2.imwrite(str(out_path), frame)
            saved += 1
    cap.release()
    return saved


def process_split(df: pd.DataFrame, split_name: str, val_fraction: float = 0.0):
    """
    Process a CSV split (train or test).
    For the train split optionally carve out a val sub-split.
    """
    if val_fraction > 0:
        train_part, val_part = train_test_split(
            df, test_size=val_fraction, stratify=df["label"],
            random_state=RANDOM_SEED
        )
        splits = [("train", train_part), ("val", val_part)]
    else:
        splits = [(split_name, df)]

    for folder_name, part_df in splits:
        print(f"\nProcessing → {folder_name}  ({len(part_df)} videos)")
        for _, row in tqdm(part_df.iterrows(), total=len(part_df), desc=folder_name):
            video_rel = str(row["video_path"]).strip().lstrip("/")
            # The CSV may contain paths like "abuse/Abuse001.mp4" or just "Abuse001.mp4"
            video_file = DATASET_ROOT / video_rel
            if not video_file.exists():
                # Fall back: search by filename inside the class directory
                label = str(row["label"]).strip().lower()
                video_file = DATASET_ROOT / label / Path(video_rel).name
            if not video_file.exists():
                continue  # skip missing videos silently

            label_clean = str(row["label"]).strip().lower().replace(" ", "_")
            out_dir = FRAMES_DIR / folder_name / label_clean
            extract_frames(video_file, out_dir, n_frames=FRAMES_PER_VID)


# ── Run extraction ─────────────────────────────────────────────────────────────
print("Extracting frames from TRAIN videos (+ carving out VAL)…")
process_split(train_df, split_name="train", val_fraction=VAL_SPLIT)

print("\nExtracting frames from TEST videos…")
process_split(test_df, split_name="test", val_fraction=0.0)

# ── Report ────────────────────────────────────────────────────────────────────
for split in ["train", "val", "test"]:
    split_dir = FRAMES_DIR / split
    if split_dir.exists():
        n = sum(1 for _ in split_dir.rglob("*.jpg"))
        print(f"{split:6s} : {n} frames")

## 6. Verify Extracted Frames

Sanity-check: display a random grid of extracted frames.

In [ ]:
def show_sample_frames(frames_root: Path, split: str = "train", n_cols: int = 5, n_rows: int = 3):
    """Display a random grid of sample frames from the given split."""
    all_images = list((frames_root / split).rglob("*.jpg"))
    if not all_images:
        print(f"No frames found in {frames_root / split}")
        return

    sample = random.sample(all_images, min(n_cols * n_rows, len(all_images)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 2.5))
    axes = axes.flatten()
    for ax, img_path in zip(axes, sample):
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(img_path.parent.name, fontsize=8)
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    plt.suptitle(f"Sample frames — {split}", fontsize=14)
    plt.tight_layout()
    plt.show()


show_sample_frames(FRAMES_DIR, split="train")

# Class counts per split
for split in ["train", "val", "test"]:
    split_dir = FRAMES_DIR / split
    if not split_dir.exists():
        continue
    print(f"\n{split.upper()} class frame counts:")
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            n = len(list(cls_dir.glob("*.jpg")))
            print(f"  {cls_dir.name:<20s}: {n}")

## 7. Data YAML Configuration

For `task=classify` YOLO reads only the `path` key pointing to the root of the
`train/` and `val/` subdirectories.

In [ ]:
# Derive class names from the extracted train directory
train_cls_dirs = sorted([
    d.name for d in (FRAMES_DIR / "train").iterdir()
    if d.is_dir()
]) if (FRAMES_DIR / "train").exists() else CLASSES

NC = len(train_cls_dirs)
print(f"Number of classes : {NC}")
print(f"Class names       : {train_cls_dirs}")

# Build YAML content
# ultralytics classify expects:  path = <root>, train = train, val = val
data_yaml = f"""# UCF Crime — YOLO26 Classification Dataset
path: {FRAMES_DIR.as_posix()}   # root dir containing train/ and val/
train: train
val: val
test: test

nc: {NC}
names: {train_cls_dirs}
"""

yaml_path = FRAMES_DIR / "ucf_crime.yaml"
yaml_path.write_text(data_yaml)
print(f"\ndata.yaml written to: {yaml_path}")
print("\nContents:")
print(data_yaml)

## 8. Model Initialisation

Load the **YOLOv26 nano** base weights and print a summary.

In [ ]:
model = YOLO(YOLO26_WEIGHTS)   # loads yolo26n.pt

# Print model info
model.info(verbose=True)
print(f"\nModel task : {model.task}")
print(f"Model type : {type(model.model).__name__}")

## 9. Model Training

Train the YOLOv26 model using the classification task on the extracted frames.

In [ ]:
results = model.train(
    task         = "classify",           # video-level classification (no bbox annotations)
    data         = str(yaml_path),       # path to ucf_crime.yaml
    epochs       = EPOCHS,               # default: 50  — increase for better accuracy
    imgsz        = IMG_SIZE,             # default: 224 — standard size for classification
    batch        = BATCH_SIZE,           # default: 16  — reduce to 8 if you run out of VRAM
    device       = DEVICE,               # default: 0 (GPU) or "cpu" if no CUDA GPU found
    project      = str(RUNS_DIR),        # output root: custom_models/ucf_crime_runs/
    name         = "yolo26_ucf_crime",   # experiment sub-folder name
    patience     = 20,                   # early-stop if val accuracy doesn't improve for 20 epochs
    lr0          = 0.01,                 # initial learning rate
    lrf          = 0.01,                 # final learning rate (lr0 * lrf)
    momentum     = 0.937,                # SGD momentum
    weight_decay = 0.0005,               # L2 regularisation
    warmup_epochs= 3,                    # epochs to ramp up LR from 0 → lr0
    cos_lr       = True,                 # cosine LR decay schedule
    augment      = True,                 # enable built-in augmentation
    degrees      = 10.0,                 # random rotation ± 10°
    flipud       = 0.0,                  # vertical flip probability (0 = off)
    fliplr       = 0.5,                  # horizontal flip probability
    hsv_h        = 0.015,                # hue jitter
    hsv_s        = 0.7,                  # saturation jitter
    hsv_v        = 0.4,                  # brightness jitter
    workers      = 4,                    # dataloader worker threads
    verbose      = True,
    exist_ok     = True,                 # overwrite previous run with same name
)

# ── Quick summary of what was used ────────────────────────────────────────────
print("\nTraining complete.")
print(f"  epochs    : {EPOCHS}")
print(f"  imgsz     : {IMG_SIZE}")
print(f"  batch     : {BATCH_SIZE}")
print(f"  device    : {DEVICE}")
print(f"Best weights: {RUNS_DIR}/yolo26_ucf_crime/weights/best.pt")

## 10. Training Results Visualisation

In [ ]:
results_csv = RUNS_DIR / "yolo26_ucf_crime" / "results.csv"

if not results_csv.exists():
    print(f"results.csv not found at: {results_csv}")
else:
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()
    print("Available metrics columns:", df_res.columns.tolist())

    # ── Dynamically find loss and accuracy columns ─────────────────────────
    train_loss_cols = [c for c in df_res.columns if "train" in c.lower() and "loss" in c.lower()]
    val_loss_cols   = [c for c in df_res.columns if "val"   in c.lower() and "loss" in c.lower()]
    acc_top1_cols   = [c for c in df_res.columns if "acc" in c.lower() and "top1" in c.lower()]
    acc_top5_cols   = [c for c in df_res.columns if "acc" in c.lower() and "top5" in c.lower()]

    epochs_axis = range(1, len(df_res) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    ax = axes[0]
    for col in train_loss_cols:
        ax.plot(epochs_axis, df_res[col], label=col)
    for col in val_loss_cols:
        ax.plot(epochs_axis, df_res[col], linestyle="--", label=col)
    ax.set_title("Loss over epochs")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=7)
    ax.grid(True)

    # Accuracy
    ax = axes[1]
    for col in acc_top1_cols:
        ax.plot(epochs_axis, df_res[col], label=col)
    for col in acc_top5_cols:
        ax.plot(epochs_axis, df_res[col], linestyle="--", label=col)
    ax.set_title("Accuracy over epochs")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend(fontsize=7)
    ax.grid(True)

    plt.tight_layout()
    plt.show()

    # Print best epoch summary
    if acc_top1_cols:
        best_epoch = df_res[acc_top1_cols[0]].idxmax()
        print(f"\nBest epoch (top-1 acc): {best_epoch + 1}")
        print(df_res.iloc[best_epoch].to_string())

In [ ]:
# ── Display YOLO-generated plots (confusion matrix, results grid) ─────────────
plots_dir = RUNS_DIR / "yolo26_ucf_crime"
plot_files = list(plots_dir.glob("*.png")) + list(plots_dir.glob("*.jpg"))

if plot_files:
    fig, axes = plt.subplots(1, len(plot_files), figsize=(6 * len(plot_files), 5))
    if len(plot_files) == 1:
        axes = [axes]
    for ax, pf in zip(axes, plot_files):
        img = mpimg.imread(str(pf))
        ax.imshow(img)
        ax.set_title(pf.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No plot images found yet — run training first.")

## 11. Model Validation & Evaluation

In [ ]:
# Load best weights for validation
best_weights = RUNS_DIR / "yolo26_ucf_crime" / "weights" / "best.pt"

if not best_weights.exists():
    print(f"best.pt not found at {best_weights}. Run training first.")
else:
    best_model = YOLO(str(best_weights))

    # Validate on the val split
    val_results = best_model.val(
        data    = str(yaml_path),
        split   = "val",
        imgsz   = IMG_SIZE,
        device  = DEVICE,
        verbose = True,
    )

    print("\n── Validation Metrics ──────────────────────────────")
    print(f"  Top-1 Accuracy : {val_results.top1:.4f}")
    print(f"  Top-5 Accuracy : {val_results.top5:.4f}")

In [ ]:
# ── Sample predictions visualisation ────────────────────────────────────────
val_frames = list((FRAMES_DIR / "val").rglob("*.jpg"))
if val_frames:
    sample_preds = random.sample(val_frames, min(10, len(val_frames)))

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    axes = axes.flatten()

    for ax, img_path in zip(axes, sample_preds):
        pred = best_model.predict(str(img_path), verbose=False, imgsz=IMG_SIZE)[0]
        top_class = pred.probs.top1         # integer class index
        confidence = pred.probs.top1conf.item()
        true_label = img_path.parent.name

        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        color = "green" if train_cls_dirs[top_class] == true_label else "red"
        ax.set_title(
            f"True : {true_label}\nPred : {train_cls_dirs[top_class]} ({confidence:.2f})",
            fontsize=7, color=color
        )
        ax.axis("off")

    for ax in axes[len(sample_preds):]:
        ax.axis("off")

    plt.suptitle("Sample Predictions (green = correct, red = wrong)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No val frames found.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Gather ground-truth and predictions over the entire val set
y_true, y_pred = [], []

for img_path in tqdm(val_frames, desc="Evaluating val set"):
    pred = best_model.predict(str(img_path), verbose=False, imgsz=IMG_SIZE)[0]
    y_true.append(train_cls_dirs.index(img_path.parent.name)
                  if img_path.parent.name in train_cls_dirs else -1)
    y_pred.append(pred.probs.top1)

# Filter out any -1 (unknown class in dir)
valid = [(t, p) for t, p in zip(y_true, y_pred) if t != -1]
y_true_f, y_pred_f = zip(*valid) if valid else ([], [])

print(classification_report(y_true_f, y_pred_f, target_names=train_cls_dirs, zero_division=0))

cm = confusion_matrix(y_true_f, y_pred_f)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=train_cls_dirs,
            yticklabels=train_cls_dirs, cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Val Set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 12. Export & Save Model

Export the best weights to ONNX and TorchScript, then copy them to the
dedicated `custom_models/ucf_crime_weights/` directory.

In [ ]:
if not best_weights.exists():
    print("best.pt not found — run training first.")
else:
    export_model = YOLO(str(best_weights))

    # Export to ONNX
    print("Exporting to ONNX…")
    onnx_path = export_model.export(format="onnx", imgsz=IMG_SIZE, dynamic=True, simplify=True)
    print(f"  ONNX saved  : {onnx_path}")

    # Export to TorchScript
    print("Exporting to TorchScript…")
    ts_path = export_model.export(format="torchscript", imgsz=IMG_SIZE)
    print(f"  TorchScript : {ts_path}")

    # Copy weights to MODEL_OUT_DIR
    MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
    for src in [best_weights, Path(onnx_path), Path(ts_path)]:
        if src and src.exists():
            dst = MODEL_OUT_DIR / src.name
            shutil.copy2(src, dst)
            print(f"  Copied → {dst}")

    print(f"\nAll model files saved in: {MODEL_OUT_DIR}")

## 13. Quick Inference Test

Run inference on a single unseen video file to verify the deployed model works end-to-end.

In [ ]:
def classify_video(video_path: str | Path, model_path: str | Path,
                   class_names: list[str], n_frames: int = 16,
                   imgsz: int = 224) -> dict:
    """
    Classify a video by majority-vote over uniformly sampled frames.
    Returns a dict with predicted class, confidence, and per-class scores.
    """
    clf = YOLO(str(model_path))
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, min(n_frames, total), dtype=int)

    class_scores = np.zeros(len(class_names))
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pred = clf.predict(frame_rgb, verbose=False, imgsz=imgsz)[0]
        class_scores += pred.probs.data.cpu().numpy()
    cap.release()

    class_scores /= (class_scores.sum() + 1e-9)
    top_idx = int(np.argmax(class_scores))
    return {
        "predicted_class": class_names[top_idx],
        "confidence"      : float(class_scores[top_idx]),
        "all_scores"      : {cls: float(sc) for cls, sc in zip(class_names, class_scores)},
    }


# ── Test on a random video from the dataset ──────────────────────────────────
test_videos = list(DATASET_ROOT.rglob("*.mp4")) + list(DATASET_ROOT.rglob("*.avi"))

if test_videos and best_weights.exists():
    test_vid = random.choice(test_videos)
    print(f"Testing on: {test_vid}")
    result = classify_video(test_vid, best_weights, train_cls_dirs)
    print(f"\nTrue label       : {test_vid.parent.name}")
    print(f"Predicted class  : {result['predicted_class']}")
    print(f"Confidence       : {result['confidence']:.4f}")
    print("\nAll class scores:")
    for cls, score in sorted(result["all_scores"].items(), key=lambda x: -x[1]):
        bar = "█" * int(score * 40)
        print(f"  {cls:<20s} {score:.4f}  {bar}")
else:
    print("Either no test videos found or best.pt is missing — run previous cells first.")